# 04 — Comparative Analysis: Semantic vs Lexical Search

This notebook evaluates the effectiveness of semantic search against traditional keyword-based (lexical) search using two levels of lexical baselines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from sentence_transformers import SentenceTransformer, util

df = pd.read_csv('aletheia_filtered.csv')
embeddings = np.load('embeddings_filtered.npy')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
device = model.device

### 1. Define Keyword Maps

In [ ]:
# Basic anchor words
strict_map = {
    "Vacinas causam doenças autoimunes": ["autoimune"],
    "Vacinas contêm microchips ou substâncias ocultas": ["microchip", "chip"],
    "Governos escondem efeitos colaterais das vacinas": ["governo", "esconde"],
    "A imunidade natural é superior à imunidade vacinal": ["natural", "superior"],
    "Vacinas foram desenvolvidas rápido demais e não são seguras": ["rápido", "experimental"]
}

# Expanded dictionary (synonyms and technical terms)
expanded_map = {
    "Vacinas causam doenças autoimunes": ["autoimune", "autoimunidade", "lupus", "esclerose", "mimetismo", "spike"],
    "Vacinas contêm microchips ou substâncias ocultas": ["microchip", "chip", "grafeno", "nanobot", "5g", "nanotecnologia", "magnetismo"],
    "Governos escondem efeitos colaterais das vacinas": ["governo", "esconde", "oculta", "omite", "mentira", "anvisa", "segredo"],
    "A imunidade natural é superior à imunidade vacinal": ["natural", "superior", "vitamina d", "zinco", "inata"],
    "Vacinas foram desenvolvidas rápido demais e não são seguras": ["rápido", "experimental", "recorde", "testes", "segurança", "fase 3"]
}


### 2. Result Comparison (Top 100)

In [ ]:
comp_list = []
corpus_t = torch.from_numpy(embeddings).to(device)

for narr in strict_map.keys():
    # 1. Semantic Search (Top 100)
    q_emb = model.encode(narr, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, corpus_t)[0]
    sem_idx = set(torch.topk(scores, k=100).indices.tolist())
    
    # 2. Strict Lexical Search
    strict_pat = '|'.join(strict_map[narr])
    hits_strict = set(df[df['text_content'].str.contains(strict_pat, case=False, na=False)].index)
    
    # 3. Expanded Lexical Search
    expanded_pat = '|'.join(expanded_map[narr])
    hits_expanded = set(df[df['text_content'].str.contains(expanded_pat, case=False, na=False)].index)
    
    comp_list.append({
        'narrative': narr,
        'strict_lexical': len(sem_idx.intersection(hits_strict)),
        'expanded_lexical': len(sem_idx.intersection(hits_expanded)),
        'semantic_only': 100 - len(sem_idx.intersection(hits_expanded))
    })

results_df = pd.DataFrame(comp_list)
results_df

### 3. Visualizing the Semantic Gap

In [ ]:
import textwrap
results_df['narrative_wrapped'] = results_df['narrative'].apply(lambda x: textwrap.fill(x, 22))
ax = results_df.plot(kind='bar', x='narrative_wrapped', figsize=(12,6), width=0.8)
plt.title('Comparação: Abordagem Léxica vs Semântica')
plt.ylabel('Número de Mensagens')
plt.xlabel('')
plt.xticks(rotation=0, ha='center', fontsize=9)
plt.legend(['Busca Léxica Exata', 'Busca Léxica Expandida', 'Exclusivo da Busca Semântica'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
import os
if not os.path.exists('figures'):
    os.makedirs('figures')
plt.savefig('figures/04_lexical_comparison.png', dpi=300)
plt.show()


### 4. Qualitative Analysis: 'Hidden' Messages

In [ ]:
target = "Vacinas contêm microchips ou substâncias ocultas"
q_emb = model.encode(target, convert_to_tensor=True)
scores = util.cos_sim(q_emb, corpus_t)[0]
sem_indices = torch.topk(scores, k=100).indices.tolist()

pat = '|'.join(expanded_map[target])
lex_indices = set(df[df['text_content'].str.contains(pat, case=False, na=False)].index)

print(f'Examples missed by lexical search (even expanded) for: {target}')
count = 0
for i in sem_indices:
    if i not in lex_indices:
        text = df.iloc[i]['text_content'][:180].replace('\n', ' ')
        print(f'- {text}...')
        count += 1
    if count >= 3: break